# **🇹🇭 Constituency OCR: Using Google Gemini API (High Accuracy)**

This notebook uses the highly capable **Gemini 1.5 Flash** (or Pro) model via API to extract vote counts directly from the election form images. Using an API is often much more accurate and faster for complex handwritten Thai OCR than a local model, provided the competition allows internet access during inference or you're running it locally.

### 🔑 How to securely add your API Key in Kaggle:
1. In the Kaggle Notebook top menu, click **Add-ons** > **Secrets**.
2. Click **Add a new secret**.
3. Set the **Label** as `GEMINI_API_KEY`.
4. Paste your Gemini API key (from [Google AI Studio](https://aistudio.google.com/)) into the **Value** field.
5. Click **Save** and make sure the toggle next to the secret is turned **ON**.
6. Make sure **Internet** is turned **ON** in the Notebook options (right sidebar).

In [10]:
!pip install -q google-generativeai rapidfuzz textdistance pandas pillow tqdm

In [11]:
import os
import re
import json
import glob
import time
import textdistance
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from rapidfuzz import process, fuzz
import google.generativeai as genai

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    API_KEY = user_secrets.get_secret("GEMINI_API_KEY")
    print("✅ Kaggle Secret 'GEMINI_API_KEY' found!")
except Exception as e:
    API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
    if API_KEY == "YOUR_API_KEY_HERE":
        print("❌ WARNING: API Key not found! The code will fail with 400/404 errors.")
    else:
        print("✅ API Key loaded from environment variables!")

genai.configure(api_key=API_KEY)

# Dynamically find the best available Flash model to avoid 404 Not Found errors
AVAILABLE_MODEL = 'gemini-1.5-flash' # Fallback
try:
    models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
    flash_models = [m for m in models if 'flash' in m.lower()]
    if flash_models:
        # Prefer gemini-2.5-flash or gemini-2.0-flash if 1.5 is deprecated
        flash_models.sort(reverse=True) 
        AVAILABLE_MODEL = flash_models[0].replace('models/', '')
        print(f"✅ Dynamically selected model: {AVAILABLE_MODEL}")
    else:
        print("⚠️ No flash models found in your project! You might get a 404.")
except Exception as e:
    print(f"⚠️ Could not list models (check your API Key!): {e}")

model = genai.GenerativeModel(AVAILABLE_MODEL)


Kaggle Secret Found!


## 1. Helper Functions to Load Images and Clean Text

In [12]:
DATA_DIR = '/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/data'
IMG_DIR = os.path.join(DATA_DIR, 'images')
TEMPLATE_PATH = os.path.join(DATA_DIR, 'submission_template.csv')

sub_df = pd.read_csv(TEMPLATE_PATH)
doc_ids = sub_df['doc_id'].unique()

def get_images_for_doc(doc_id):
    """Reads all PNG pages for a given doc_id."""
    images = []
    base_img = os.path.join(IMG_DIR, f"{doc_id}.png")
    if os.path.exists(base_img):
        images.append(base_img)
    
    page = 2
    while True:
        page_img = os.path.join(IMG_DIR, f"{doc_id}_page{page}.png")
        if os.path.exists(page_img):
            images.append(page_img)
            page += 1
        else:
            break
    return [Image.open(img) for img in images]

def clean_vote_string(s):
    """Removes non-digits and converts Thai numerals to Arabic."""
    thai_to_arabic = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
    s = str(s).translate(thai_to_arabic)
    s = re.sub(r'\D', '', s)
    return s if s else "0"

## 2. API Extraction Pipeline
We can pass images simultaneously to Gemini to read the full context across the 5 pages.

In [13]:
def extract_votes_gemini(doc_id, parties_list):
    images = get_images_for_doc(doc_id)
    if not images:
        return {p: "0" for p in parties_list}
    
    parties_str = "\n".join([f"- {p}" for p in parties_list])
    
    prompt = f"""
    You are an expert OCR AI specializing in reading Thai political election result documents. 
    The user has attached {len(images)} pages relating to a single constituency's vote tally.
    
    Find the vote counts for the following political parties across all pages:
    {parties_str}
    
    Rules:
    1. Locate the exact row for each party name.
    2. Extract the numerical vote count for that party.
    3. Reply strictly and ONLY with a well-formatted JSON dictionary mapping the party name to the votes as strings.
    
    Example Output:
    {{
      "PartyA": "1532"
    }}
    """
    
    try:
        # Instead of pushing PIL Images inline (which can cause 404/400 API errors if size is too big),
        # we will pass the PIL images directly, but if the image is massive, let's resize it slightly.
        resized_images = []
        for img in images:
            img.thumbnail((1600, 1600)) # Prevent payload overload
            resized_images.append(img)
            
        payload = [prompt] + resized_images
        
        time.sleep(2) # Prevent rate limits
        response = model.generate_content(payload, generation_config={'temperature': 0.1})
        output = response.text.strip()
        
        if output.startswith("```json"):
            output = output.split("```json")[1].split("```")[0]
        elif output.startswith("```"):
            output = output.split("```")[1].split("```")[0]
            
        extracted_map = json.loads(output)
        
        final_map = {}
        for k, v in extracted_map.items():
            final_map[k] = clean_vote_string(str(v))
            
        return final_map

    except Exception as e:
        print(f"API Error on {doc_id}: {e}")
        return {p: "0" for p in parties_list}


## 3. Local Evaluation on Ground Truths

In [14]:
def evaluate_on_samples():
    label_files = glob.glob(os.path.join(DATA_DIR, "sample_labels", "*.json"))
    total_dist = 0
    total_rows = 0
    
    print(f"Evaluating on {len(label_files)} sample ground truths...")
    
    for lpath in label_files:
        with open(lpath, 'r', encoding='utf-8') as f:
            gt_data = json.load(f)
            
        doc_id = os.path.basename(lpath).replace('.json', '')
        gt_map = {item['party']: str(item['votes']) for item in gt_data['results']}
        
        pred_map = extract_votes_gemini(doc_id, list(gt_map.keys()))
        
        for party, gt_votes in gt_map.items():
            pred_votes = pred_map.get(party, "0")
            if party not in pred_map and len(pred_map) > 0:
                 best_match = process.extractOne(party, list(pred_map.keys()), scorer=fuzz.ratio)
                 if best_match and best_match[1] > 75:
                     pred_votes = pred_map[best_match[0]]
            
            dist = textdistance.levenshtein(gt_votes, pred_votes)
            total_dist += dist
            total_rows += 1
            
    if total_rows > 0:
        print(f"Gemini API Mean Levenshtein Distance: {total_dist / total_rows:.4f}")
        
# Uncomment to run local test
# evaluate_on_samples()

## 4. Final Submission

In [15]:
results_list = []

for doc_id in tqdm(doc_ids[:5], desc="Processing Test Documents API"):
    parties_to_find = sub_df[sub_df['doc_id'] == doc_id]['party_name'].tolist()
    
    extracted_map = extract_votes_gemini(doc_id, parties_to_find)
    
    for party in parties_to_find:
        # เปลี่ยนจาก "0" (ข้อความ) เป็น 0 (ตัวเลข)
        votes = extracted_map.get(party, 0) 
        
        if party not in extracted_map and len(extracted_map) > 0:
             best_match = process.extractOne(party, list(extracted_map.keys()), scorer=fuzz.ratio)
             if best_match and best_match[1] > 75:
                 votes = extracted_map[best_match[0]]
                 
        # ดึงค่าที่ clean แล้วออกมา
        cleaned_vote = clean_vote_string(str(votes))
        
        results_list.append({
            'doc_id': doc_id,
            'party_name': party,
            # แปลงเป็น int ก่อนเก็บเข้า list (ดัก Error กรณี string ว่างเปล่าไว้ด้วย)
            'votes': int(cleaned_vote) if cleaned_vote else 0
        })
        
for res in results_list:
    mask = (sub_df['doc_id'] == res['doc_id']) & (sub_df['party_name'] == res['party_name'])
    # ตอนนี้ res['votes'] เป็นตัวเลข (int) แล้ว จะไม่เกิดกล่องแดง
    sub_df.loc[mask, 'votes'] = res['votes']

sub_df.to_csv('submission_gemini.csv', index=False)
print("Submission successfully saved as submission_gemini.csv!")
sub_df.head()

Processing Test Documents API:   0%|          | 0/5 [00:00<?, ?it/s]

API Error on constituency_10_1: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
API Error on constituency_10_10: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
API Error on constituency_10_11: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
API Error on constituency_10_12: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found f

,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,0
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,0
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,0
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,0
4,constituency_10_1_5,constituency_10_1,5,พลวัต,0
